# Trend Scanning for Labeling Financial Time Series

**Course:** BUSI70575 — Systematic Trading Strategies with ML
**Source:** `refs/programming-session-sol/Solution_Programming_Session_1.ipynb` (applied in Session 5)

Trend scanning (López de Prado) is a **labeling** method. For each observation it looks across a range
of horizons, fits a straight line to the future (or past) prices over each horizon, and keeps the
horizon whose linear trend is **most statistically significant**. The label is the *sign* of that
trend; its *strength* is the regression **t-value**.

This notebook derives the core number (the t-value of an OLS slope), states the exact algorithm, and
**verifies every number against the course code**. Run top-to-bottom with the **`stml`** kernel.

---

## 1. Intuition

* Fix a point in time $t_0$. Ask: *if I look forward $L$ bars, is there a trend, and how strong is it?*
* A **straight line** $p_t \approx \beta_0 + \beta_1 t$ fit by least squares has slope $\hat\beta_1$.
  The slope alone is not enough — a steep slope on noisy data is not a real trend. We need the
  **t-statistic** $t_{\hat\beta_1} = \hat\beta_1 / \mathrm{SE}(\hat\beta_1)$, the slope measured
  *relative to its uncertainty*.
* Trend scanning tries **many horizons** $L$ and keeps the one with the largest $|t|$ — the clearest
  trend the data shows around $t_0$. The label is $\operatorname{sign}(t)$: **+1** up-trend, **−1** down-trend.
* `look_forward=True` looks into the future (used to build *labels*); `look_forward=False` looks back
  (safe to use for *features* — no look-ahead).

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm1     # the course imports statsmodels under the alias sm1

np.random.seed(42)

## 2. The core number — t-value of an OLS slope

Over a window of $n$ consecutive prices we regress price on a time index $x = 0,1,\dots,n-1$:

$$p_x = \beta_0 + \beta_1 x + \varepsilon_x.$$

**Slope (least squares).** With $\bar x,\bar p$ the means,
$$\hat\beta_1=\frac{\sum_x (x-\bar x)(p_x-\bar p)}{\sum_x (x-\bar x)^2}=\frac{S_{xy}}{S_{xx}},\qquad
\hat\beta_0=\bar p-\hat\beta_1\bar x.$$

**Standard error of the slope.** With residuals $\hat\varepsilon_x = p_x-\hat\beta_0-\hat\beta_1 x$ and
$\mathrm{SSR}=\sum_x\hat\varepsilon_x^2$, the residual variance uses $n-2$ degrees of freedom:
$$\hat\sigma^2=\frac{\mathrm{SSR}}{n-2},\qquad \mathrm{SE}(\hat\beta_1)=\sqrt{\frac{\hat\sigma^2}{S_{xx}}}.$$

**t-value.**
$$\boxed{\;t_{\hat\beta_1}=\dfrac{\hat\beta_1}{\mathrm{SE}(\hat\beta_1)}\;}$$

This is exactly what `statsmodels` returns as `ols.tvalues[1]` inside the course's `tValLinR`.

### Worked example (by hand)

Prices $p=[10,11,13,12,15]$ at $x=[0,1,2,3,4]$, so $n=5$.

| quantity | value |
|---|---|
| $\bar x,\ \bar p$ | $2,\ 12.2$ |
| $S_{xx}=\sum (x-\bar x)^2$ | $4+1+0+1+4=10$ |
| $S_{xy}=\sum (x-\bar x)(p-\bar p)$ | $4.4+1.2+0-0.2+5.6=11.0$ |
| $\hat\beta_1=S_{xy}/S_{xx}$ | $11.0/10=\mathbf{1.1}$ |
| $\hat\beta_0=\bar p-\hat\beta_1\bar x$ | $12.2-2.2=\mathbf{10.0}$ |
| residuals $\hat\varepsilon$ | $[0,\,-0.1,\,0.8,\,-1.3,\,0.6]$ |
| $\mathrm{SSR}$ | $0+0.01+0.64+1.69+0.36=2.70$ |
| $\hat\sigma^2=\mathrm{SSR}/(n-2)$ | $2.70/3=0.90$ |
| $\mathrm{SE}(\hat\beta_1)=\sqrt{0.90/10}$ | $\sqrt{0.09}=0.30$ |
| $t=\hat\beta_1/\mathrm{SE}$ | $1.1/0.3=\mathbf{3.667}$ |

The next cells reproduce this from scratch **and** check it against `tValLinR`.

In [ ]:
# --- Verbatim from Solution_Programming_Session_1.ipynb (cell 4) ---
def tValLinR(close):
    """
    Calculate the t-value and coefficients of the slope from a linear regression of the time series.

    Parameters:
    - close (pd.Series): A pandas series of closing prices.

    Returns:
    - tuple: (t-value of the slope coefficient, coefficients of the regression)
    """
    x = np.ones((close.shape[0], 2))
    x[:, 1] = np.arange(close.shape[0])
    ols = sm1.OLS(close, x).fit()
    return ols.tvalues[1], ols.params

In [ ]:
y = np.array([10, 11, 13, 12, 15], dtype=float)
n = len(y)
x = np.arange(n, dtype=float)

xbar, ybar = x.mean(), y.mean()
Sxx = ((x - xbar) ** 2).sum()
Sxy = ((x - xbar) * (y - ybar)).sum()
beta1 = Sxy / Sxx
beta0 = ybar - beta1 * xbar

resid = y - (beta0 + beta1 * x)
SSR = (resid ** 2).sum()
sigma2 = SSR / (n - 2)                 # dof = n - 2
SE_beta1 = np.sqrt(sigma2 / Sxx)
t_stat = beta1 / SE_beta1

print(f"beta1 = {beta1:.4f}    beta0 = {beta0:.4f}")
print(f"SSR   = {SSR:.4f}    sigma^2 = {sigma2:.4f}")
print(f"SE(beta1) = {SE_beta1:.4f}    t = {t_stat:.4f}")

# (a) match the hand-worked numbers
assert np.isclose(beta1, 1.1)
assert np.isclose(SE_beta1, 0.3)
assert np.isclose(t_stat, 11 / 3)

# (b) match the course function tValLinR (statsmodels)
t_course, params = tValLinR(y)
print(f"\ntValLinR -> t = {t_course:.4f},  params (b0, b1) = {np.round(params, 4)}")
assert np.isclose(t_course, t_stat)
assert np.allclose(params, [beta0, beta1])
print("OK: hand-worked == from-scratch == statsmodels")

## 3. The `trend_labels` algorithm

For every index $t_0$ in the series:

1. For each horizon $h$ in the span (`range(*observation_span)`), take the window of $h{+}1$ points
   (`look_forward`: $t_0,\dots,t_0{+}h$) and compute its t-value with `tValLinR`.
2. Pick the horizon with the **largest absolute t-value**: $h^\star=\arg\max_h |t_h|$.
3. Record `bin` $=\operatorname{sign}(t_{h^\star})$, `tVal` $=t_{h^\star}$, `windowSize` $=h^\star$, and
   `t1` = the timestamp ending the winning window (the label's *event end* / vertical barrier).

Points without room for the largest horizon are skipped. Finally the raw t-values are **capped** to
`tMax = min(20, var(tVal))` to tame outliers — this caps *magnitude only*; `bin` and `windowSize` were
already fixed in step 2 (before capping).

In [ ]:
# --- Verbatim from Solution_Programming_Session_1.ipynb (cell 16) ---
def trend_labels(price_series, observation_span, look_forward=True):
    """
    Generate labels for segments of a time series based on the trend (slope) over a specified observation span.
    Returns a DataFrame with columns ['t1', 'tVal', 'bin', 'windowSize'].
    """
    out = pd.DataFrame(index=price_series.index, columns=['t1', 'tVal', 'bin', 'windowSize'])
    hrzns = range(*observation_span)

    for idx in price_series.index:
        tval_dict = {}
        iloc0 = price_series.index.get_loc(idx)
        if look_forward and iloc0 > len(price_series) - observation_span[1]:
            continue
        if not look_forward and iloc0 < observation_span[1]:
            continue

        for hrzn in hrzns:
            if look_forward:
                dt1 = idx
                dt2 = min(iloc0 + hrzn, len(price_series) - 1)
                dt2 = price_series.index[dt2]
            else:
                dt1 = max(iloc0 - hrzn, 0)
                dt1 = price_series.index[dt1]
                dt2 = idx
            df1 = price_series.loc[dt1:dt2]
            tval_dict[hrzn], _ = tValLinR(df1.values)

        max_hrzn = max(tval_dict, key=lambda x: abs(tval_dict[x]))
        if look_forward:
            max_dt1 = min(iloc0 + max_hrzn, len(price_series) - 1)
            max_dt1 = price_series.index[max_dt1]
        else:
            max_dt1 = max(iloc0 - max_hrzn, 0)
            max_dt1 = price_series.index[max_dt1]

        out.loc[idx, ['t1', 'tVal', 'bin', 'windowSize']] = \
            max_dt1, tval_dict[max_hrzn], np.sign(tval_dict[max_hrzn]), max_hrzn

    if isinstance(price_series.index, pd.DatetimeIndex):
        out['t1'] = pd.to_datetime(out['t1'])
    out['bin'] = pd.to_numeric(out['bin'], downcast='signed')

    tValueVariance = out['tVal'].values.var()
    tMax = min(20, tValueVariance)
    out.loc[out['tVal'] > tMax, 'tVal'] = tMax
    out.loc[out['tVal'] < -tMax, 'tVal'] = -tMax
    return out.dropna(subset=['bin'])

### Trace one point by hand

A 16-point up-then-down series with span `(3, 6)` (horizons 3, 4, 5; windows of 4, 5, 6 points). We run
the course function, then independently reproduce the horizon choice at index 0 and confirm the
`bin`/`windowSize` agree (these are decided *before* the t-value cap, so the check is exact).

In [ ]:
prices = pd.Series([100, 101, 103, 102, 104, 106, 109, 111,
                    110, 108, 107, 104, 103, 101, 100,  98], dtype=float)
span = (3, 6)                                   # horizons 3, 4, 5
labels = trend_labels(prices, span, look_forward=True)
print(labels, "\n")

# Reproduce the horizon selection for index 0 (pre-cap):
i0 = 0
tvals = {}
for h in range(*span):
    window = prices.iloc[i0:i0 + h + 1].values   # h+1 points: iloc0 .. iloc0+h
    tvals[h], _ = tValLinR(window)
best_h = max(tvals, key=lambda h: abs(tvals[h]))
print("t by horizon at index 0:", {h: round(v, 3) for h, v in tvals.items()})
print(f"argmax|t| -> horizon {best_h}, t = {tvals[best_h]:.3f}, bin = {np.sign(tvals[best_h]):+.0f}")

assert labels.loc[i0, "windowSize"] == best_h
assert labels.loc[i0, "bin"] == np.sign(tvals[best_h])
print("OK: trend_labels row 0 matches the hand-traced argmax")

## 4. Using the labels

* **`bin`** is the classification target a meta-model learns to predict (Session 5 trains RF / XGBoost
  on features to predict this trend label).
* **`t1`** marks where each label's event ends — the natural *vertical barrier* for sample weighting and
  for avoiding overlapping-label leakage.
* **`tVal`** is a confidence / strength score — bright points below are strong trends, dark points weak.

The plot recreates Session 1's trend-scanning scatter on a synthetic up→down→up path.

In [ ]:
rng = np.random.default_rng(7)
trend = np.concatenate([np.linspace(0, 16, 70),
                        np.linspace(16, 3, 80),
                        np.linspace(3, 22, 60)])
price = 100 + trend + rng.normal(0, 1.0, size=trend.size)
series = pd.Series(price)

lab = trend_labels(series, (5, 20), look_forward=True)
print("label counts:")
print(lab["bin"].value_counts())

ev = series.loc[lab.index]
plt.figure(figsize=(11, 4))
sc = plt.scatter(ev.index, ev.values, c=lab["tVal"].astype(float), s=14, cmap="viridis")
plt.colorbar(sc, label="t-value (capped)")
plt.xlabel("time index"); plt.ylabel("price")
plt.title("Trend-scanning labels (color = t-value)")
plt.tight_layout(); plt.show()

## 5. Exam traps & gotchas

* **Look-ahead is built in (and that is fine *for labels*).** `look_forward=True` uses *future* prices —
  fine for building targets, never for features. Use `look_forward=False` if you turn a trend reading
  into a feature.
* **t-value, not slope.** The label uses the **t-value**. A big slope with big noise can have a small
  $|t|$ (weak label); a gentle but clean slope can have a large $|t|$.
* **Degrees of freedom $n-2$.** A window of $n$ points has $n-2$ dof; you need $n\ge 3$ for a finite SE
  ($n\ge 2$ just to fit the line). Very short windows give unstable t-values.
* **The `min(20, var(tVal))` cap is quirky.** It compares 20 to a *variance*; on low-variance series the
  cap can shrink reported `tVal`s sharply. It does **not** change `bin`/`windowSize`.
* **Horizon span flips labels.** Session 1 Q3 shows the *same* date reading +1 over a long horizon and
  −1 over a short one — the label only means something together with its `windowSize`.

## Source pointers

| What | File | Cell |
|---|---|---|
| `tValLinR` (t-value of slope) | `Solution_Programming_Session_1.ipynb` | 4 |
| `plot_trend_lines` (horizon visual) | `Solution_Programming_Session_1.ipynb` | 6 |
| `trend_labels` (the algorithm) | `Solution_Programming_Session_1.ipynb` | 16 |
| BTC application + tVal scatter | `Solution_Programming_Session_1.ipynb` | 18–21 |
| Trend labels feeding a meta-model | `Solution_Programming_Session_5.ipynb` | — |

Reference: M. López de Prado, *Advances in Financial Machine Learning* — trend-scanning labels.